# 📖 Notebook 3: Surge Pricing

When everyone wants a ride at the same time (concert ending, rainstorm, New Year's Eve), there aren't enough drivers. Uber's solution: **raise the price** until some riders decide to wait and more drivers are incentivized to come online.

This is surge pricing — a dynamic multiplier applied to the base fare based on the supply/demand ratio in a geographic zone.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to calculate supply (available drivers) and demand (ride requests) per zone
- How to compute a surge multiplier from the supply/demand ratio
- How zone-based pricing works with PostGIS spatial queries
- How to apply surge to fare estimates

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/uber
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import random
import json

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "uber_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL + PostGIS")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Why Surge Pricing?

Imagine a Friday night in downtown San Francisco:

- **50 people** open the app wanting a ride
- **10 drivers** are available in the area

Without surge: 10 riders get matched, 40 wait forever. Drivers have no incentive to come to the area.

With surge (2.5×): 
- Some riders decide it's too expensive and take the bus → demand drops to 25
- Drivers in nearby areas see the surge and drive to downtown → supply rises to 20
- More riders get served, drivers earn more, the market balances

**Surge pricing is a real-time market mechanism.** The multiplier goes up when demand > supply and comes back down when they balance.

In [ ]:
# Let's look at our surge zones

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        id,
        zone_name,
        ST_Y(center::geometry) AS latitude,
        ST_X(center::geometry) AS longitude,
        radius_km,
        current_demand,
        current_supply,
        surge_multiplier
    FROM surge_zones
    ORDER BY id;
""")

print("📊 Surge Zones in San Francisco:")
print(f"{'ID':<4} {'Zone':<22} {'Lat':>8} {'Lng':>10} {'Radius':>7} {'Demand':>7} {'Supply':>7} {'Surge':>6}")
print("-" * 78)
for row in cur.fetchall():
    print(f"{row[0]:<4} {row[1]:<22} {row[2]:>8.4f} {row[3]:>10.4f} {row[4]:>5.1f}km {row[5]:>7} {row[6]:>7} {row[7]:>5.2f}×")

conn.close()
print()
print("💡 All zones start with demand=0, supply=0, surge=1.0×")
print("   We'll simulate traffic and watch surge change.")

## 📊 Calculating the Surge Multiplier

The core formula is simple:

```
demand_supply_ratio = demand / supply

If ratio <= 1.0  → surge = 1.0× (no surge — enough drivers)
If ratio  > 1.0  → surge grows with the ratio, capped at some max
```

Real Uber uses much more sophisticated models, but the core idea is the same: **price rises when demand outpaces supply**.

In [ ]:
def calculate_surge(demand, supply, max_surge=5.0):
    """
    Calculate the surge multiplier based on demand and supply.
    
    Args:
        demand: Number of ride requests in the zone
        supply: Number of available drivers in the zone
        max_surge: Maximum surge multiplier (cap)
    
    Returns:
        Surge multiplier (1.0 = no surge)
    """
    if supply == 0:
        # No drivers at all — max surge to attract drivers
        return max_surge if demand > 0 else 1.0
    
    ratio = demand / supply
    
    if ratio <= 1.0:
        return 1.0  # enough drivers, no surge
    
    # Surge grows with the square root of the ratio
    # This makes surge increase quickly at first, then slow down
    # Example: ratio=2 → 1.4×, ratio=4 → 2.0×, ratio=9 → 3.0×
    surge = ratio ** 0.5
    
    # Round to nearest 0.25 (Uber-style pricing increments)
    surge = round(surge * 4) / 4
    
    return min(surge, max_surge)


# Show how surge changes with different demand/supply ratios
print("📊 Surge Multiplier Examples:")
print(f"{'Demand':>7} {'Supply':>7} {'Ratio':>7} {'Surge':>7}")
print("-" * 32)

examples = [
    (5, 10),    # more drivers than requests
    (10, 10),   # balanced
    (15, 10),   # slight shortage
    (20, 10),   # 2:1 ratio
    (40, 10),   # 4:1 ratio
    (90, 10),   # 9:1 ratio (concert ending)
    (50, 0),    # no drivers at all
]

for demand, supply in examples:
    ratio = demand / supply if supply > 0 else "∞"
    surge = calculate_surge(demand, supply)
    ratio_str = f"{ratio:.1f}" if isinstance(ratio, float) else ratio
    print(f"{demand:>7} {supply:>7} {ratio_str:>7} {surge:>6.2f}×")

## 🗺️ Zone-Based Supply & Demand Counting

To calculate surge per zone, we need to count:
1. **Supply**: How many available drivers are within each zone's radius
2. **Demand**: How many ride requests originated from each zone recently

For supply, we use PostGIS to count drivers within each zone's geographic boundary.  
For demand, we count recent ride requests (tracked in Redis for speed).

In [ ]:
# First, let's load driver locations into Redis for this demo

r = get_redis()
conn = get_db()
cur = conn.cursor()

r.delete("drivers:locations", "drivers:available")

cur.execute("""
    SELECT d.id, d.status,
           ST_X(dl.location::geometry) AS lng,
           ST_Y(dl.location::geometry) AS lat
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id;
""")
for row in cur.fetchall():
    did, status, lng, lat = row
    r.geoadd("drivers:locations", (lng, lat, f"driver:{did}"))
    if status == "available":
        r.sadd("drivers:available", f"driver:{did}")

conn.close()
print(f"✅ Loaded {r.zcard('drivers:locations')} drivers into Redis")
print(f"   {r.scard('drivers:available')} are available")

In [ ]:
def count_supply_in_zone(zone_lng, zone_lat, radius_km):
    """Count available drivers within a zone using Redis Geo."""
    r = get_redis()
    nearby = r.geosearch(
        name="drivers:locations",
        longitude=zone_lng, latitude=zone_lat,
        radius=radius_km, unit="km"
    )
    available = r.smembers("drivers:available")
    return len([d for d in nearby if d in available])


def simulate_demand(zone_id, count):
    """Simulate ride requests in a zone by incrementing a Redis counter."""
    r = get_redis()
    # In production, this would be incremented each time a ride is requested
    # The counter resets every pricing window (e.g., every 2 minutes)
    r.set(f"zone:{zone_id}:demand", count, ex=120)  # expires in 2 min


def get_demand(zone_id):
    """Get the current demand count for a zone."""
    r = get_redis()
    val = r.get(f"zone:{zone_id}:demand")
    return int(val) if val else 0


# Simulate different demand levels across zones
# Downtown: Friday night rush. SoMa: convention ending. Others: normal.
demand_simulation = {
    1: 35,  # Downtown/Financial — happy hour
    2: 50,  # SoMa/Convention — big event ending
    3: 8,   # Mission — normal evening
    4: 5,   # Castro — quiet
    5: 12,  # Marina — moderate
    6: 25,  # SFO Airport — flight arrivals
}

for zone_id, demand in demand_simulation.items():
    simulate_demand(zone_id, demand)

print("✅ Simulated demand across 6 zones")

In [ ]:
# Now calculate surge for each zone!

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT id, zone_name,
           ST_X(center::geometry) AS lng,
           ST_Y(center::geometry) AS lat,
           radius_km
    FROM surge_zones ORDER BY id;
""")
zones = cur.fetchall()

print("🔥 Surge Pricing Calculation:")
print(f"{'Zone':<22} {'Supply':>7} {'Demand':>7} {'Ratio':>7} {'Surge':>7} {'Status':>12}")
print("-" * 68)

for zone_id, name, lng, lat, radius_km in zones:
    supply = count_supply_in_zone(lng, lat, float(radius_km))
    demand = get_demand(zone_id)
    surge = calculate_surge(demand, supply)
    ratio = f"{demand/supply:.1f}" if supply > 0 else "∞"
    
    if surge >= 3.0:
        status = "🔴 EXTREME"
    elif surge >= 2.0:
        status = "🟠 HIGH"
    elif surge > 1.0:
        status = "🟡 ACTIVE"
    else:
        status = "🟢 NORMAL"
    
    print(f"{name:<22} {supply:>7} {demand:>7} {ratio:>7} {surge:>6.2f}× {status:>12}")
    
    # Update the surge in the database
    cur.execute("""
        UPDATE surge_zones 
        SET current_demand = %s, current_supply = %s, 
            surge_multiplier = %s, updated_at = NOW()
        WHERE id = %s;
    """, (demand, supply, surge, zone_id))

conn.commit()
conn.close()

print()
print("💡 In production, this calculation runs every 1-2 minutes per zone.")
print("   Zones with more demand than supply get a surge multiplier.")

## 💰 Applying Surge to Fare Estimates

When a rider requests a fare estimate, the system:
1. Calculates the base fare from distance and time
2. Determines which surge zone the pickup is in
3. Multiplies the base fare by the zone's surge multiplier

Let's build this end-to-end.

In [ ]:
def estimate_fare(pickup_lng, pickup_lat, dropoff_lng, dropoff_lat):
    """
    Calculate a fare estimate with surge pricing.
    
    Returns a dict with base fare, surge multiplier, and final estimate.
    """
    conn = get_db()
    cur = conn.cursor()
    
    # Step 1: Calculate distance between pickup and dropoff (in km)
    cur.execute("""
        SELECT ST_Distance(
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography
        ) / 1000.0 AS distance_km;
    """, (pickup_lng, pickup_lat, dropoff_lng, dropoff_lat))
    distance_km = float(cur.fetchone()[0])
    
    # Step 2: Calculate base fare
    # Simple pricing: $2.50 base + $1.50/km + $0.25/min (assume 2 min/km in city)
    base_charge = 2.50
    per_km = 1.50
    per_min = 0.25
    estimated_minutes = distance_km * 2  # rough estimate: 2 min/km in city
    base_fare = base_charge + (distance_km * per_km) + (estimated_minutes * per_min)
    
    # Step 3: Find the surge zone containing the pickup location
    cur.execute("""
        SELECT zone_name, surge_multiplier
        FROM surge_zones
        WHERE ST_DWithin(
            center,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
            radius_km * 1000  -- convert km to meters
        )
        ORDER BY ST_Distance(
            center,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography
        )
        LIMIT 1;
    """, (pickup_lng, pickup_lat, pickup_lng, pickup_lat))
    
    zone_row = cur.fetchone()
    zone_name = zone_row[0] if zone_row else "No Zone"
    surge = float(zone_row[1]) if zone_row else 1.0
    
    # Step 4: Apply surge
    estimated_fare = round(base_fare * surge, 2)
    
    conn.close()
    
    return {
        "distance_km": round(distance_km, 2),
        "estimated_minutes": round(estimated_minutes, 0),
        "base_fare": round(base_fare, 2),
        "surge_zone": zone_name,
        "surge_multiplier": surge,
        "estimated_fare": estimated_fare,
    }


# Test with different pickup locations
# All going to SFO Airport (-122.3790, 37.6213)
dropoff = (-122.3790, 37.6213)

pickups = [
    ("Downtown/Financial", -122.4000, 37.7900),
    ("SoMa (convention)",  -122.4000, 37.7800),
    ("Mission District",   -122.4200, 37.7600),
    ("Castro (quiet)",     -122.4350, 37.7600),
]

print("💰 Fare Estimates: Various Pickups → SFO Airport")
print("=" * 80)

for label, plng, plat in pickups:
    fare = estimate_fare(plng, plat, *dropoff)
    print(f"\n  📍 Pickup: {label}")
    print(f"     Distance:   {fare['distance_km']} km")
    print(f"     Base fare:  ${fare['base_fare']:.2f}")
    print(f"     Surge zone: {fare['surge_zone']} ({fare['surge_multiplier']}×)")
    print(f"     Final fare: ${fare['estimated_fare']:.2f}", end="")
    if fare['surge_multiplier'] > 1.0:
        extra = fare['estimated_fare'] - fare['base_fare']
        print(f"  (⚡ +${extra:.2f} surge)")
    else:
        print("  (no surge)")

## 🔄 Watching Surge Change Over Time

Surge pricing isn't static — it recalculates every 1-2 minutes. Let's simulate demand changing and watch surge respond.

In [ ]:
# Simulate 3 pricing windows for the SoMa/Convention zone
# Scenario: A big tech conference just ended

zone_id = 2  # SoMa/Convention
zone_supply = 3  # 3 available drivers in this area

# Demand over time: spike then gradual decline
demand_over_time = [
    ("6:00 PM — Conference ends", 60),
    ("6:02 PM — Peak exodus",     80),
    ("6:04 PM — Surge attracts drivers", 50),
    ("6:06 PM — Demand falling",  30),
    ("6:08 PM — Some wait for Muni", 15),
    ("6:10 PM — Back to normal",  5),
]

# Assume supply increases as drivers respond to surge
supply_over_time = [3, 3, 5, 8, 10, 10]

print("📊 SoMa/Convention Zone — Surge Over Time")
print("=" * 70)
print(f"{'Time':<35} {'Demand':>7} {'Supply':>7} {'Surge':>7} {'Visual'}")
print("-" * 70)

for i, (label, demand) in enumerate(demand_over_time):
    supply = supply_over_time[i]
    surge = calculate_surge(demand, supply)
    bar = "🔥" * int(surge)
    print(f"{label:<35} {demand:>7} {supply:>7} {surge:>6.2f}× {bar}")

print()
print("💡 Notice how:")
print("   1. Surge spikes when demand >> supply")
print("   2. High surge attracts more drivers (supply increases)")
print("   3. Higher supply + lower demand → surge comes back down")
print("   4. This creates a self-balancing market")

## 🧹 Cleanup

In [ ]:
r = get_redis()
# Clean up demand counters and driver data
for key in r.keys("zone:*:demand"):
    r.delete(key)
r.delete("drivers:locations", "drivers:available")

# Reset surge zones to defaults
conn = get_db()
cur = conn.cursor()
cur.execute("""
    UPDATE surge_zones 
    SET current_demand = 0, current_supply = 0, surge_multiplier = 1.00;
""")
conn.commit()
conn.close()

print("🧹 Cleaned up Redis keys and reset surge zones")

## 📚 Summary

### Key Takeaways

1. **Surge pricing balances supply and demand** — it's a real-time market mechanism, not just "charging more"
2. **The formula is simple**: `surge = f(demand / supply)` — capped at a maximum multiplier
3. **Zone-based pricing** uses PostGIS to determine which zone a pickup falls in
4. **Surge is self-correcting** — high prices attract more drivers AND reduce demand
5. **Recalculate frequently** — every 1-2 minutes per zone to respond to changing conditions

### How This Fits in a System Design Interview

Surge pricing shows you understand:
- **Supply/demand economics** in distributed systems
- **Geographic partitioning** — different zones have independent pricing
- **Real-time aggregation** — counting requests per zone per time window
- **Combining Redis + PostGIS** — fast counters + spatial queries

### Next Up

In **Notebook 4**, we'll tackle the **trip lifecycle** — managing ride state from request to completion, with distributed locking to prevent double-assignment of drivers.